# Importing Library

In [20]:
import tensorflow as tf
import matplotlib.pyplot as plt
import numpy as np
from tensorflow.keras.layers import Input, Dropout, Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.metrics import SparseCategoricalAccuracy
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from tensorflow.keras.layers import RandomFlip, RandomRotation, RandomZoom , RandomTranslation , RandomContrast

# Importing Dataset

In [7]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 16
SEED = 123
train_ds = tf.keras.utils.image_dataset_from_directory(
    "data/train",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='int',
    shuffle=True,
    validation_split=0.2,
    subset="training",
    seed=SEED
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    "data/train",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='int',
    shuffle=True,
    validation_split=0.2,
    subset="validation",
    seed=SEED
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    "data/test",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='int',
    shuffle=True
)

Found 6799 files belonging to 3 classes.
Using 5440 files for training.
Found 6799 files belonging to 3 classes.
Using 1359 files for validation.
Found 2278 files belonging to 3 classes.


# Data Preprocessing

In [27]:
data_augmentation = tf.keras.Sequential([
    RandomRotation(factor=(-0.025, 0.025)),
    RandomFlip(mode="horizontal"),
    RandomBrightness(factor=0.1)
], name="data_augmentation")

# Model Creation

In [38]:
def build_model(input_shape=IMG_SIZE + (3,)):
    base_model = tf.keras.applications.ResNet50(
        include_top=False,
        input_shape=input_shape,
        weights="imagenet"
    )
    base_model.trainable = False

    inputs = Input(shape=input_shape)
    x = data_augmentation(inputs) 
    x = base_model(x, training=False)
    x = GlobalAveragePooling2D()(x)
    outputs = Dense(3, activation="softmax")(x)

    model = Model(inputs, outputs, name="Human_Emotion_Detection_Model")
    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=[SparseCategoricalAccuracy(name="accuracy")]
    )
    return model, base_model

model, base_model = build_model()
model.summary()

Model: "Human_Emotion_Detection_Model"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_24 (InputLayer)     │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ data_augmentation (Sequential)  │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ resnet50 (Functional)           │ (None, 7, 7, 2048)     │    23,587,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_9      │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 3)              │         6,147 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 23,593,859 (90.00 MB)

 Trainable params: 6,147 (24.01 KB)

 Non-trainable params: 23,587,712 (89.98 MB)

# Training The Model

In [39]:
history1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5,
    callbacks=[
        EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)
    ]
)

Epoch 1/5
340/340 ━━━━━━━━━━━━━━━━━━━━ 27s 68ms/step - accuracy: 0.6344 - loss: 0.8201 - val_accuracy: 0.6681 - val_loss: 0.7956
Epoch 2/5
340/340 ━━━━━━━━━━━━━━━━━━━━ 19s 56ms/step - accuracy: 0.7189 - loss: 0.6751 - val_accuracy: 0.6586 - val_loss: 0.8085
Epoch 3/5
340/340 ━━━━━━━━━━━━━━━━━━━━ 22s 65ms/step - accuracy: 0.7443 - loss: 0.6182 - val_accuracy: 0.7601 - val_loss: 0.6077
Epoch 4/5
340/340 ━━━━━━━━━━━━━━━━━━━━ 19s 55ms/step - accuracy: 0.7675 - loss: 0.5782 - val_accuracy: 0.7277 - val_loss: 0.6574
Epoch 5/5
340/340 ━━━━━━━━━━━━━━━━━━━━ 22s 64ms/step - accuracy: 0.7748 - loss: 0.5522 - val_accuracy: 0.7631 - val_loss: 0.6078


# Fine Tuning

In [ ]:
print("Base model has", len(base_model.layers), "layers.")

fine_tune_at = 110  
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False
for layer in base_model.layers[fine_tune_at:]:
    layer.trainable = True


model.compile(
    optimizer=tf.keras.optimizers.Adam(5e-6),
    loss="sparse_categorical_crossentropy",
    metrics=[SparseCategoricalAccuracy(name="accuracy")]
)

history2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=[
        ModelCheckpoint("best__model.keras", save_best_only=True, monitor="val_loss"),
        EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True),
        ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-7)
    ]
)

Base model has 175 layers.
Epoch 1/10
340/340 ━━━━━━━━━━━━━━━━━━━━ 51s 118ms/step - accuracy: 0.7491 - loss: 0.6118 - val_accuracy: 0.7859 - val_loss: 0.5384 - learning_rate: 1.0000e-05
Epoch 2/10
340/340 ━━━━━━━━━━━━━━━━━━━━ 39s 114ms/step - accuracy: 0.8322 - loss: 0.4279 - val_accuracy: 0.8013 - val_loss: 0.4958 - learning_rate: 1.0000e-05
Epoch 3/10
340/340 ━━━━━━━━━━━━━━━━━━━━ 42s 123ms/step - accuracy: 0.8678 - loss: 0.3488 - val_accuracy: 0.8168 - val_loss: 0.4584 - learning_rate: 1.0000e-05
Epoch 4/10
340/340 ━━━━━━━━━━━━━━━━━━━━ 39s 115ms/step - accuracy: 0.8934 - loss: 0.2840 - val_accuracy: 0.8337 - val_loss: 0.4364 - learning_rate: 1.0000e-05
Epoch 5/10
340/340 ━━━━━━━━━━━━━━━━━━━━ 39s 116ms/step - accuracy: 0.9121 - loss: 0.2415 - val_accuracy: 0.8418 - val_loss: 0.4210 - learning_rate: 1.0000e-05
Epoch 6/10
340/340 ━━━━━━━━━━━━━━━━━━━━ 39s 114ms/step - accuracy: 0.9347 - loss: 0.1959 - val_accuracy: 0.8455 - val_loss: 0.4185 - learning_rate: 1.0000e-05
Epoch 7/10
340/340 

# Evaluate

In [43]:
model.evaluate(test_ds)

143/143 ━━━━━━━━━━━━━━━━━━━━ 6s 41ms/step - accuracy: 0.8507 - loss: 0.4483


[0.4483174681663513, 0.8507462739944458]